In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml kailash-align kailash-kaizen

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# M6 Ollama bootstrap — local LLMs, no API keys
# On Colab: installs + starts Ollama and pulls this lesson's models.
# Locally: verifies the daemon is running and models are pulled.
# ══════════════════════════════════════════════════════════════════
_LESSON_MODELS = ['llama3.2:3b']
import sys, os, time, shutil, subprocess
import httpx

_OLLAMA_BASE_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
_IN_COLAB = 'google.colab' in sys.modules

def _daemon_up(timeout_s=1.0):
    try:
        r = httpx.get(f'{_OLLAMA_BASE_URL}/api/tags', timeout=timeout_s)
        r.raise_for_status()
        return r.json()
    except Exception:
        return None

if _IN_COLAB and shutil.which('ollama') is None:
    print('[ollama] installing binary …')
    subprocess.run(
        'curl -fsSL https://ollama.com/install.sh | sh',
        shell=True, check=True,
    )

if _IN_COLAB and _daemon_up() is None:
    print('[ollama] starting daemon …')
    subprocess.Popen(
        'nohup ollama serve > /tmp/ollama.log 2>&1 &',
        shell=True,
    )
    deadline = time.monotonic() + 30
    while time.monotonic() < deadline and _daemon_up() is None:
        time.sleep(0.5)

_tags = _daemon_up(timeout_s=2.0)
if _tags is None:
    raise RuntimeError(
        f'Ollama daemon not reachable at {_OLLAMA_BASE_URL}. '
        'On Colab: re-run this cell. Locally: run `ollama serve`.'
    )

_have = {m.get('name','').split(':',1)[0] for m in _tags.get('models', [])}
for _model in _LESSON_MODELS:
    _fam = _model.split(':', 1)[0]
    if _fam in _have:
        continue
    if _IN_COLAB:
        print(f'[ollama] pulling {_model} (one-time) …')
        subprocess.run(['ollama', 'pull', _model], check=True)
    else:
        raise RuntimeError(
            f'Model {_model!r} not pulled. Run: ollama pull {_model}'
        )

print(
    '✓ Ollama ready at', _OLLAMA_BASE_URL,
    '— models:', _LESSON_MODELS,
)



In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp06
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


def create_visualizer():
    """Return a ModelVisualizer with the P2 experimental notice acknowledged.

    kailash-ml 2.2.2 emits ExperimentalWarning (a UserWarning) at
    ModelVisualizer construction; exercises run under warnings-as-errors,
    so the notice is acknowledged narrowly here — the ONE construction site
    every course file should use.
    """
    import warnings as _warnings

    from kailash_ml import ModelVisualizer
    from kailash_ml._decorators import ExperimentalWarning as _EW

    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore", _EW)
        return ModelVisualizer()

# ── SDK-internal deprecation nag filter (strict-gate hygiene) ────────────
# torch 2.12's ONNX exporter warns that dynamic_axes is "not recommended"
# under dynamo=True — a deprecation notice inside OnnxBridge's internals,
# not a correctness signal, and fatal under warnings-as-errors. Acknowledged
# here (kailash_helpers is imported by every shared.mlfpNN package, so every
# technique file gets it); message-matched, everything else still errors.
import warnings as _warnings

_warnings.filterwarnings(
    "ignore", message=r".*dynamic_axes.*not recommended.*", category=UserWarning
)
# matplotlib's Agg backend (forced in headless/script runs) nags
# "FigureCanvasAgg is non-interactive, and thus cannot be shown" on every
# plt.show() — a backend notice, not a correctness signal.
_warnings.filterwarnings(
    "ignore", message=r".*non-interactive.*cannot be shown.*", category=UserWarning
)

def hdb_storey_range_expr(column: str = "storey_range"):
    """Polars expression normalising HDB storey_range letter-O typos.

    The raw HDB resale file has letter-O typos ("O4 TO 06", "1O TO 12",
    "28 TO 3O"); read a letter O next to a digit as zero, leaving "TO"
    alone. Single home for the rule (P2): M2 ex_8 and M3 ex_5/07 both
    normalise at the use-site.
    """
    import polars as pl

    return (
        pl.col(column)
        .str.replace_all(r"\bO(\d)", "0${1}")
        .str.replace_all(r"(\d)O\b", "${1}0")
    )



# ── shared/mlfp06/_ollama_bootstrap.py ──
"""Ollama bootstrap for MLFP06 — local LLMs, no API keys, real calls only.

Why this module exists
----------------------
M6 trains, prompts, evaluates, and deploys real LLMs. Routing every call
through OpenAI:

  * forces every learner to hold an API key
  * burns budget every time a notebook reruns a cell
  * makes "did the LLM actually fire?" invisible (a stubbed `cost=$0`
    looks identical to a successful call)

This module switches every M6 LLM call to a locally-running Ollama daemon
that the student installs once and re-uses across every lesson.

Contract
--------
* :func:`make_delegate` — the only sanctioned way to construct a Kaizen
  ``Delegate`` for M6. Wraps the Ollama adapter with sensible defaults and
  forces ``budget_usd=None`` (Kaizen mis-prices Ollama as $3/$15-per-Mtok;
  budget tracking against a free local provider is meaningless).
* :func:`make_embedder` — the only sanctioned way to construct an embedding
  adapter for M6 RAG (lesson 6.4).
* :func:`preflight_ollama` — pre-flight reachability + model-presence check.
  Raises :class:`OllamaUnreachableError` with an actionable message. Call
  this at the top of every notebook / script before constructing a
  Delegate.
* :data:`LESSON_MODELS` — manifest of which models each lesson needs.
  Consumed by ``scripts/generate_selfcontained_notebook.py`` to inject the
  Colab bootstrap cell.

Per ``rules/zero-tolerance.md`` Rule 2: there is no silent stub fallback
in this module. If Ollama is not reachable, we raise. The previous
``run_delegate(...) -> ("unknown", 0.0, latency)`` fallback is BLOCKED.

Per ``rules/framework-first.md``: every LLM call goes through Kaizen
``Delegate``. Raw ``httpx`` calls to ``/api/generate`` are BLOCKED for
generation paths; ``httpx`` is used here only for the daemon-reachability
preflight.
"""

import logging
import os
import shutil
import subprocess
import sys
import time
from typing import Any

import httpx
from dotenv import load_dotenv

logger = logging.getLogger(__name__)

# ════════════════════════════════════════════════════════════════════════
# Configuration — env-driven per rules/env-models.md
# ════════════════════════════════════════════════════════════════════════

load_dotenv()


def _env(key: str, default: str) -> str:
    val = os.environ.get(key)
    return val if val else default


OLLAMA_BASE_URL: str = _env("OLLAMA_BASE_URL", "http://localhost:11434")
"""Where the Ollama daemon listens. Local default; Colab uses the same."""

DEFAULT_CHAT_MODEL: str = _env("OLLAMA_CHAT_MODEL", "llama3.2:3b")
"""Tool-capable instruct model. Allowlisted by Kaizen for function calling."""

DEFAULT_EMBED_MODEL: str = _env("OLLAMA_EMBED_MODEL", "nomic-embed-text")
"""Compact embedding model — ~270MB, 768-dim. Used by M6 RAG (lesson 6.4)."""

DEFAULT_FT_BASE_MODEL: str = _env("OLLAMA_FT_BASE_MODEL", "qwen2.5:0.5b")
"""Tiny base model for fine-tuning demos (lessons 6.2, 6.3). Fits T4."""


# ════════════════════════════════════════════════════════════════════════
# Lesson → models manifest (used by generator + preflight)
# ════════════════════════════════════════════════════════════════════════


LESSON_MODELS: dict[str, list[str]] = {
    # 6.1 Prompting — chat only
    "ex_1": [DEFAULT_CHAT_MODEL],
    # 6.2 Fine-tuning — base for LoRA + chat for evaluation
    "ex_2": [DEFAULT_FT_BASE_MODEL, DEFAULT_CHAT_MODEL],
    # 6.3 Alignment (DPO/GRPO) — base + chat for judge
    "ex_3": [DEFAULT_FT_BASE_MODEL, DEFAULT_CHAT_MODEL],
    # 6.4 RAG — chat + embeddings
    "ex_4": [DEFAULT_CHAT_MODEL, DEFAULT_EMBED_MODEL],
    # 6.5 Agents (ReAct) — tool-capable chat
    "ex_5": [DEFAULT_CHAT_MODEL],
    # 6.6 Multi-agent + MCP — tool-capable chat
    "ex_6": [DEFAULT_CHAT_MODEL],
    # 6.7 PACT governance — chat for governed Delegate
    "ex_7": [DEFAULT_CHAT_MODEL],
    # 6.8 Nexus deployment — chat behind the API
    "ex_8": [DEFAULT_CHAT_MODEL],
}


# ════════════════════════════════════════════════════════════════════════
# Errors
# ════════════════════════════════════════════════════════════════════════


class OllamaUnreachableError(RuntimeError):
    """Daemon not running, daemon unreachable, or required model not pulled.

    The error message MUST be actionable: it tells the user the exact
    command to run to fix the situation. We never wrap a low-level
    ``httpx.ConnectError`` and re-raise without translating it.
    """


# ════════════════════════════════════════════════════════════════════════
# Preflight — reachability + model presence
# ════════════════════════════════════════════════════════════════════════


def _running_in_colab() -> bool:
    return "google.colab" in sys.modules


def _list_pulled_families(*, timeout_s: float = 2.0) -> set[str]:
    """Hit /api/tags and return the set of pulled model families (no tag).

    Returns ``{"llama3.2", "qwen2.5"}`` for a daemon with
    ``llama3.2:3b`` and ``qwen2.5:0.5b`` pulled. The model family is
    everything before the first colon — we match against family rather
    than the full ``family:tag`` because pulling ``llama3.2:3b-instruct``
    or ``llama3.2:3b-instruct-q4_K_M`` should still satisfy a request
    for the ``llama3.2`` family.
    """
    url = f"{OLLAMA_BASE_URL.rstrip('/')}/api/tags"
    try:
        r = httpx.get(url, timeout=timeout_s)
        r.raise_for_status()
    except (httpx.HTTPError, httpx.ConnectError) as exc:
        raise OllamaUnreachableError(
            f"ollama daemon not reachable at {OLLAMA_BASE_URL}. "
            f"Fix: open a terminal and run `ollama serve`. "
            f"On macOS the desktop app starts the daemon automatically. "
            f"Underlying: {type(exc).__name__}: {exc}"
        ) from exc
    payload = r.json() or {}
    families: set[str] = set()
    for entry in payload.get("models", []):
        name = entry.get("name") or entry.get("model") or ""
        if not name:
            continue
        families.add(name.split(":", 1)[0])
    return families


def preflight_ollama(
    *,
    required_models: list[str] | None = None,
    auto_pull: bool | None = None,
    timeout_s: float = 2.0,
) -> None:
    """Verify the daemon is reachable and the required models are pulled.

    Args:
        required_models: Models that must be present, e.g.
            ``["llama3.2:3b", "nomic-embed-text"]``. Family-matched (the
            ``:tag`` portion is informational; any pulled tag for the
            same family satisfies the requirement).
        auto_pull: If ``True``, attempt ``ollama pull <model>`` for any
            missing model via subprocess. Defaults to ``True`` on Colab
            (where the bootstrap cell has just installed the binary)
            and ``False`` locally (where pulling is the student's job
            and surprise downloads are unwelcome).
        timeout_s: HTTP timeout for the reachability check. The default
            is intentionally short — if the daemon is alive it answers
            within 100ms.

    Raises:
        OllamaUnreachableError: Daemon unreachable, or a required model
            is missing after pull (or auto_pull was disabled).
    """
    if auto_pull is None:
        auto_pull = _running_in_colab()

    have = _list_pulled_families(timeout_s=timeout_s)
    needed = required_models or []
    missing = [m for m in needed if m.split(":", 1)[0] not in have]

    if missing and auto_pull:
        for model in missing:
            _pull_model(model)
        # Re-check
        have = _list_pulled_families(timeout_s=timeout_s)
        missing = [m for m in needed if m.split(":", 1)[0] not in have]

    if missing:
        cmds = "\n  ".join(f"ollama pull {m}" for m in missing)
        raise OllamaUnreachableError(
            f"Required model(s) not pulled: {missing}.\n"
            f"Fix — run in a terminal:\n  {cmds}"
        )

    logger.info(
        "ollama.preflight.ok",
        extra={
            "base_url": OLLAMA_BASE_URL,
            "models_present": sorted(have),
            "required": needed,
        },
    )


def _pull_model(model: str) -> None:
    """Run ``ollama pull <model>`` as a subprocess. Streams to stdout."""
    if shutil.which("ollama") is None:
        raise OllamaUnreachableError(
            f"`ollama` binary not on PATH — cannot pull {model}. "
            f"Install: https://ollama.com/download"
        )
    print(f"[ollama] pulling {model} (one-time download) …", flush=True)
    t0 = time.monotonic()
    proc = subprocess.run(
        ["ollama", "pull", model],
        capture_output=False,
        text=True,
    )
    elapsed = time.monotonic() - t0
    if proc.returncode != 0:
        raise OllamaUnreachableError(
            f"`ollama pull {model}` failed with exit {proc.returncode} "
            f"after {elapsed:.1f}s. Check network and disk space."
        )
    print(f"[ollama] {model} pulled in {elapsed:.1f}s", flush=True)


# ════════════════════════════════════════════════════════════════════════
# Delegate factory — the only sanctioned constructor for M6
# ════════════════════════════════════════════════════════════════════════


def make_delegate(
    *,
    model: str | None = None,
    tools: Any = None,
    system_prompt: str | None = None,
    temperature: float = 0.4,
    max_tokens: int = 4096,
    **kwargs: Any,
) -> Any:
    """Construct a Kaizen ``Delegate`` backed by the local Ollama daemon.

    This is the ONLY way M6 helpers and exercises construct a Delegate.
    Direct ``Delegate(...)`` construction in M6 code is BLOCKED — the
    cost-mispricing trap (``budget_usd`` against a free provider) and
    the silent OpenAI fallback that comes from omitting ``adapter=`` are
    the reasons.

    Args:
        model: Ollama model tag (e.g. ``"llama3.2:3b"``). Defaults to
            :data:`DEFAULT_CHAT_MODEL`.
        tools: Tool registry or list — passed through to Delegate.
            Auto-disables streaming on the Ollama adapter.
        system_prompt: Optional system prompt.
        temperature: Sampling temperature. Default 0.4.
        max_tokens: Hard cap on output length. Default 4096.
        **kwargs: Forwarded to ``Delegate(...)``. ``budget_usd`` is
            forced to ``None`` and ``api_key`` / ``base_url`` are
            BLOCKED (the adapter owns those).

    Returns:
        A configured Kaizen ``Delegate`` instance.
    """
    for blocked in ("api_key", "base_url", "budget_usd", "adapter"):
        if blocked in kwargs:
            raise ValueError(
                f"make_delegate(): {blocked!r} is owned by the bootstrap "
                f"and cannot be overridden by callers."
            )

    # Lazy imports — keep this module importable in environments that have
    # not yet installed kaizen-agents (e.g. early bootstrap cells).
    from kaizen_agents import Delegate
    from kaizen_agents.delegate.adapters.registry import get_adapter

    resolved = model or DEFAULT_CHAT_MODEL
    adapter = get_adapter(
        "ollama",
        model=resolved,
        base_url=OLLAMA_BASE_URL,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return Delegate(
        model=resolved,
        adapter=adapter,
        tools=tools,
        system_prompt=system_prompt,
        budget_usd=None,
        **kwargs,
    )


# ════════════════════════════════════════════════════════════════════════
# Embedding factory — the only sanctioned constructor for M6 RAG
# ════════════════════════════════════════════════════════════════════════


def make_embedder(*, model: str | None = None) -> Any:
    """Construct a Kaizen embedding adapter backed by Ollama.

    Args:
        model: Ollama embedding model tag. Defaults to
            :data:`DEFAULT_EMBED_MODEL`.

    Returns:
        An adapter with ``async def embed(texts: list[str]) -> list[list[float]]``.
    """
    from kaizen_agents.delegate.adapters.registry import get_embedding_adapter

    resolved = model or DEFAULT_EMBED_MODEL
    return get_embedding_adapter(
        "ollama",
        model=resolved,
        base_url=OLLAMA_BASE_URL,
    )


# ════════════════════════════════════════════════════════════════════════
# Streaming helpers — same call shape as the old `run_delegate`
# ════════════════════════════════════════════════════════════════════════


async def run_delegate_text(
    delegate: Any, prompt: str
) -> tuple[str, dict[str, int], float]:
    """Run a Delegate to completion, return ``(text, usage, latency_s)``.

    ``usage`` is a dict with ``prompt_tokens``, ``completion_tokens``,
    ``total_tokens`` (zero for events that don't carry usage). ``cost``
    is intentionally absent — Ollama is free, and Kaizen's ``consumed_usd``
    is mis-priced for local models.

    Event handling: the Kaizen ``TextDelta`` events expose the streamed
    chunk on ``.text`` (not ``.delta_text`` — that name is reserved for a
    different StreamEvent shape). The terminal ``TurnComplete`` event
    carries the final ``.text`` (entire response, not a delta) plus the
    ``.usage`` dict. We accumulate from ``TextDelta.text`` for streaming
    UX and fall back to ``TurnComplete.text`` if no deltas arrived.
    """
    t0 = time.perf_counter()
    response = ""
    usage: dict[str, int] = {
        "prompt_tokens": 0,
        "completion_tokens": 0,
        "total_tokens": 0,
    }
    final_text: str | None = None
    async for event in delegate.run(prompt):
        ev_type = getattr(event, "event_type", None)
        text = getattr(event, "text", None)
        if ev_type == "text_delta" and text:
            response += text
        elif ev_type == "turn_complete":
            final_text = text or final_text
            ev_usage = getattr(event, "usage", None) or {}
            for k in usage:
                if k in ev_usage:
                    usage[k] = int(ev_usage[k])
    if not response and final_text:
        response = final_text
    elapsed = time.perf_counter() - t0
    return response, usage, elapsed


# ════════════════════════════════════════════════════════════════════════
# Colab installer — used by the generator's Cell 0
# ════════════════════════════════════════════════════════════════════════


def colab_install_and_start_ollama(*, log_path: str = "/tmp/ollama.log") -> None:
    """Install the Ollama binary and start the daemon (Colab only).

    Idempotent: skips install if the binary is already on PATH and
    skips daemon start if a process is already listening on
    :data:`OLLAMA_BASE_URL`. Raises :class:`OllamaUnreachableError`
    if either step fails.

    Not for local use — locally, students run ``ollama serve`` themselves.
    """
    if not _running_in_colab():
        raise RuntimeError(
            "colab_install_and_start_ollama() is Colab-only. "
            "On a local machine, run `ollama serve` in a terminal."
        )

    # Step 1 — install if missing
    if shutil.which("ollama") is None:
        print("[colab] installing ollama …", flush=True)
        proc = subprocess.run(
            "curl -fsSL https://ollama.com/install.sh | sh",
            shell=True,
            check=False,
            capture_output=True,
            text=True,
        )
        if proc.returncode != 0:
            raise OllamaUnreachableError(
                f"ollama install failed (exit {proc.returncode}):\n"
                f"stdout: {proc.stdout[-500:]}\nstderr: {proc.stderr[-500:]}"
            )

    # Step 2 — start daemon if not already listening
    try:
        httpx.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=1.0).raise_for_status()
        print("[colab] ollama daemon already running", flush=True)
        return
    except (httpx.HTTPError, httpx.ConnectError):
        pass

    print(f"[colab] starting ollama daemon (logs → {log_path}) …", flush=True)
    subprocess.Popen(
        f"nohup ollama serve > {log_path} 2>&1 &",
        shell=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    # Step 3 — wait up to 30s for the daemon to answer /api/tags
    deadline = time.monotonic() + 30.0
    last_err: Exception | None = None
    while time.monotonic() < deadline:
        try:
            httpx.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=1.0).raise_for_status()
            print("[colab] ollama daemon ready", flush=True)
            return
        except (httpx.HTTPError, httpx.ConnectError) as exc:
            last_err = exc
            time.sleep(0.5)

    raise OllamaUnreachableError(
        f"ollama daemon failed to come up within 30s. "
        f"Last error: {type(last_err).__name__}: {last_err}. "
        f"Inspect {log_path} for details."
    )


# ── shared/mlfp06/ex_8.py ──
"""
Shared infrastructure for MLFP06 Exercise 8 — Capstone: Full Production Platform.

Contains: LLM model resolution (Ollama via the course bootstrap), MMLU
evaluation data loader, the PACT governance YAML (D/T/R = Department / Team /
Role) for the MLFP Capstone org and ``compile_capstone_governance()`` which
applies its clearances + envelopes, canonical Signature/Agent classes
(dataclass config + instance signature), ``build_capstone_stack(engine)``,
and the ``handle_qa`` router used by every technique file. ``handle_qa``
makes a REAL call to the local Ollama model — there is no offline stub; a
missing daemon raises with "start Ollama: ollama serve".

Technique-specific code (adapter loading, nexus registration, drift analysis,
compliance reporting) does NOT belong here — it lives in the per-technique
files under ``modules/mlfp06/solutions/ex_8/``.

Import from any cwd after ``uv sync``:

"""

import asyncio
import concurrent.futures
import os
import tempfile
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import TYPE_CHECKING, Any

import polars as pl
from dotenv import load_dotenv

from kaizen import InputField, OutputField, Signature
from kaizen.core.base_agent import BaseAgent
from kaizen_agents import GovernedSupervisor


if TYPE_CHECKING:  # pragma: no cover — type-only imports
    from pact import GovernanceEngine

# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
load_dotenv()


MODEL = DEFAULT_CHAT_MODEL  # resolved from OLLAMA_CHAT_MODEL by the bootstrap

if not MODEL:  # pragma: no cover — bootstrap default never returns empty
    raise EnvironmentError("OLLAMA_CHAT_MODEL must be set")

# Output + cache directories
OUTPUT_DIR = Path("outputs") / "ex8_capstone"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EVAL_CACHE_DIR = Path("data/mlfp06/mmlu")
EVAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
EVAL_CACHE_FILE = EVAL_CACHE_DIR / "mmlu_100.parquet"


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — MMLU (Massive Multitask Language Understanding)
# ════════════════════════════════════════════════════════════════════════


def load_mmlu_eval(n_rows: int = 100) -> pl.DataFrame:
    """Load MMLU evaluation data as a polars DataFrame, cached to parquet.

    Schema:
        instruction (str) — question + A/B/C/D choices as plain text
        response    (str) — correct letter (A/B/C/D)
        subject     (str) — MMLU subject area

    Returns:
        A polars DataFrame with at most ``n_rows`` shuffled MMLU questions.
    """
    if EVAL_CACHE_FILE.exists():
        print(f"Loading cached MMLU from {EVAL_CACHE_FILE}")
        return pl.read_parquet(EVAL_CACHE_FILE)

    print("Downloading cais/mmlu from HuggingFace...")
    from datasets import load_dataset

    ds = load_dataset("cais/mmlu", "all", split="validation")
    ds = ds.shuffle(seed=42).select(range(min(n_rows, len(ds))))
    rows: list[dict[str, Any]] = []
    for row in ds:
        choices = row["choices"]
        answer_idx = row["answer"]
        rows.append(
            {
                "instruction": (
                    f"{row['question']}\n\n"
                    f"A) {choices[0]}\nB) {choices[1]}\n"
                    f"C) {choices[2]}\nD) {choices[3]}"
                ),
                "response": ["A", "B", "C", "D"][answer_idx],
                "subject": row["subject"],
            }
        )
    eval_data = pl.DataFrame(rows)
    eval_data.write_parquet(EVAL_CACHE_FILE)
    print(f"Cached {eval_data.height} MMLU rows to {EVAL_CACHE_FILE}")
    return eval_data


# ════════════════════════════════════════════════════════════════════════
# TRAINED-ADAPTER DISCOVERY — what Ex 2.6 (SFT) and Ex 3.3 (DPO) wrote
# ════════════════════════════════════════════════════════════════════════
#
# kailash-align's AlignmentPipeline saves every trained adapter to
# ``<experiment_dir>/<adapter_name>/<method>/adapter/`` (PEFT format:
# adapter_config.json + adapter_model.safetensors). An ``AdapterRegistry()``
# built without a backing model registry lives in memory only, so a new
# process cannot see adapters registered by an earlier exercise. The
# capstone therefore re-discovers the adapters on disk and registers them.

ADAPTER_SEARCH_ROOTS: tuple[Path, ...] = (
    Path("outputs") / "ex2_finetuning",  # Ex 2.6 SFT experiment_dir
    Path("outputs/ex3_dpo/dpo_output"),  # Ex 3.3 DPO experiment_dir
)


def discover_trained_adapters(
    roots: tuple[Path, ...] = ADAPTER_SEARCH_ROOTS,
) -> list[dict[str, Any]]:
    """Return one dict per PEFT adapter found under ``roots``.

    Keys: adapter_name, method, adapter_path, base_model_id, rank, alpha,
    target_modules, trainable_params (counted from the safetensors file).
    """
    import json

    found: list[dict[str, Any]] = []
    for root in roots:
        for cfg_path in sorted(root.glob("**/adapter/adapter_config.json")):
            adapter_dir = cfg_path.parent
            meta = json.loads(cfg_path.read_text())
            weights = adapter_dir / "adapter_model.safetensors"
            found.append(
                {
                    "adapter_name": adapter_dir.parent.parent.name,
                    "method": adapter_dir.parent.name,
                    "adapter_path": str(adapter_dir),
                    "base_model_id": meta.get("base_model_name_or_path") or "",
                    "rank": int(meta.get("r") or 0),
                    "alpha": int(meta.get("lora_alpha") or 0),
                    "target_modules": tuple(sorted(meta.get("target_modules") or ())),
                    "trainable_params": (
                        count_safetensors_params(weights) if weights.exists() else 0
                    ),
                }
            )
    return found


def count_safetensors_params(path: str | Path) -> int:
    """Count parameters in one ``.safetensors`` file, or every shard in a dir.

    Reads tensor shapes from the file header only — no weights are loaded.
    """
    from math import prod

    from safetensors import safe_open

    path = Path(path)
    files = sorted(path.glob("*.safetensors")) if path.is_dir() else [path]
    total = 0
    for f in files:
        with safe_open(str(f), framework="pt") as handle:
            for key in handle.keys():
                total += prod(handle.get_slice(key).get_shape())
    return total


# ════════════════════════════════════════════════════════════════════════
# SHARED SIGNATURE & BASE AGENT
# ════════════════════════════════════════════════════════════════════════
#
# Canonical pattern:
#   1. `@dataclass` config carries the Ollama provider, model, base_url and
#      `use_async_llm=True` (run_async raises without it)
#   2. Signature is PASSED as an instance to `super().__init__(signature=...)`
#      — omitting the `signature=` keyword silently falls back to
#      `DefaultSignature()` and the declared output schema is ignored.


class CapstoneQASignature(Signature):
    """Answer questions with a governed, audited, confidence-scored response."""

    question: str = InputField(description="User's question")
    answer: str = OutputField(description="Detailed, grounded answer")
    confidence: float = OutputField(description="Confidence score 0-1")
    sources: list[str] = OutputField(description="Knowledge sources referenced")
    reasoning_steps: list[str] = OutputField(description="Step-by-step reasoning")


def _json_object_format() -> dict[str, str]:
    return {"type": "json_object"}


@dataclass
class CapstoneQAConfig:
    """Domain config — BaseAgent auto-converts to BaseAgentConfig.

    ``budget_limit_usd`` is the agent-level cost cap. On free local Ollama it
    never trips (there is no dollar cost); it matters with a paid provider.
    """

    llm_provider: str = "ollama"
    model: str = MODEL
    base_url: str = OLLAMA_BASE_URL
    temperature: float = 0.2
    budget_limit_usd: float = 5.0
    use_async_llm: bool = True
    response_format: dict = field(default_factory=_json_object_format)
    structured_output_mode: str = "explicit"


class CapstoneQAAgent(BaseAgent):
    """Capstone QA agent: wraps the fine-tuned model behind a typed signature."""

    def __init__(self, config: CapstoneQAConfig | None = None) -> None:
        super().__init__(
            config=config or CapstoneQAConfig(),
            signature=CapstoneQASignature(),
        )


# ════════════════════════════════════════════════════════════════════════
# PACT GOVERNANCE — shared org yaml (D/T/R = Department / Team / Role)
# ════════════════════════════════════════════════════════════════════════
#
# The MLFP Capstone org has one department (AI Services, D1) headed by the
# ML Director role (D1-R1), and three teams, each headed by an agent role:
#   qa_agent    D1-R1-T1-R1  public clearance        — customer-facing answers
#   admin_agent D1-R1-T2-R1  confidential clearance  — model lifecycle / metrics
#   audit_agent D1-R1-T3-R1  secret clearance        — compliance access
# pact's ladder: public < restricted < confidential < secret < top_secret.
# Each agent role gets an envelope defined by the ML Director.

ORG_YAML: str = """
# MLFP Capstone ML Platform — PACT Governance Definition
# D/T/R = Department / Team / Role; every agent role reports to a human head

org_id: "mlfp_capstone"
name: "MLFP Capstone ML Platform"

# One department, headed by the ML Director (a human role).
departments:
  - id: "ai_services"
    name: "AI Services"

# Three teams — one per agent workstream.
teams:
  - id: "qa_team"
    name: "Question Answering"
  - id: "ops_team"
    name: "Model Operations"
  - id: "audit_team"
    name: "Compliance Audit"

roles:
  # ── Department head (human) ──
  - id: "ml_director"
    name: "ML Director"
    heads: "ai_services"

  # ── Team heads (agents) ──
  - id: "qa_agent"
    name: "QA Agent"
    reports_to: "ml_director"
    heads: "qa_team"
  - id: "admin_agent"
    name: "Admin Agent"
    reports_to: "ml_director"
    heads: "ops_team"
  - id: "audit_agent"
    name: "Audit Agent"
    reports_to: "ml_director"
    heads: "audit_team"

# Clearances — pact levels; every agent at or below its head.
clearances:
  - role: "ml_director"
    level: "secret"
  - role: "qa_agent"
    level: "public"
  - role: "admin_agent"
    level: "confidential"
  - role: "audit_agent"
    level: "secret"

# Envelopes = delegations (defined_by -> target). Applied to the engine by
# compile_capstone_governance().
envelopes:
  - target: "qa_agent"
    defined_by: "ml_director"
    financial:
      max_spend_usd: 1.0
    operational:
      allowed_actions: ["generate_answer", "search_context"]
    communication:
      max_response_length: 2000

  - target: "admin_agent"
    defined_by: "ml_director"
    financial:
      max_spend_usd: 10.0
    operational:
      allowed_actions:
        - "generate_answer"
        - "search_context"
        - "update_model"
        - "view_metrics"
        - "monitor_drift"

  - target: "audit_agent"
    defined_by: "ml_director"
    financial:
      max_spend_usd: 50.0
    operational:
      allowed_actions:
        - "generate_answer"
        - "search_context"
        - "view_metrics"
        - "access_audit_log"
        - "generate_report"
"""


def write_org_yaml(path: str | Path | None = None) -> str:
    """Write the shared capstone org YAML to a temp file and return the path."""
    if path is None:
        path = os.path.join(tempfile.gettempdir(), "capstone_org.yaml")
    with open(path, "w") as f:
        f.write(ORG_YAML)
    return str(path)


def compile_capstone_governance(
    yaml_path: str | None = None,
) -> tuple["GovernanceEngine", Any]:
    """Load the capstone YAML, build the engine, APPLY clearances + envelopes.

    ``GovernanceEngine(loaded.org_definition)`` alone compiles only the
    structure; until ``apply_governance_specs`` runs, no envelope is attached
    and ``verify_action`` auto-approves every role (the installed default).

    Returns ``(engine, loaded)`` where ``loaded`` is the ``LoadedOrg``.
    """
    from kailash.trust.pact.yaml_resolvers import apply_governance_specs
    from pact import GovernanceEngine, load_org_yaml

    loaded = load_org_yaml(yaml_path or write_org_yaml())
    engine = GovernanceEngine(loaded.org_definition)
    apply_governance_specs(engine, loaded)
    return engine, loaded


# ════════════════════════════════════════════════════════════════════════
# BUILD_CAPSTONE_STACK — shared 3-tier GovernedSupervisor builder
# ════════════════════════════════════════════════════════════════════════
#
# Role -> tier mapping (clearance on pact's ladder):
#   qa    -> public        (low budget, narrow tools)
#   admin -> confidential  (mid budget, ops tools)
#   audit -> secret        (high budget, audit tools)
# The three tiers are SIBLING envelopes, each defined by (and no wider than)
# the ML Director — they are not a superset chain of one another.
#
# The helper also attaches a full 5-dimension `ConstraintEnvelopeConfig` to
# each agent role address via `engine.set_role_envelope(...)` (replacing the
# thinner YAML envelope), so `engine.verify_action()` enforces the tier.

_QA_ADDR = "D1-R1-T1-R1"
_ADMIN_ADDR = "D1-R1-T2-R1"
_AUDIT_ADDR = "D1-R1-T3-R1"
_DIRECTOR_ADDR = "D1-R1"


@dataclass
class CapstoneTier:
    """Metadata for one GovernedSupervisor tier in the capstone stack."""

    role: str
    address: str
    budget_usd: float
    tools: list[str]
    clearance: str  # pact clearance string (public | confidential | secret ...)
    description: str


CAPSTONE_TIERS: list[CapstoneTier] = [
    CapstoneTier(
        role="qa",
        address=_QA_ADDR,
        budget_usd=1.0,
        tools=["generate_answer", "search_context"],
        clearance="public",
        description="Retail-facing QA — narrow tools, low budget",
    ),
    CapstoneTier(
        role="admin",
        address=_ADMIN_ADDR,
        budget_usd=10.0,
        tools=[
            "generate_answer",
            "search_context",
            "update_model",
            "view_metrics",
            "monitor_drift",
        ],
        clearance="confidential",
        description="Model ops — update + metrics + drift",
    ),
    CapstoneTier(
        role="audit",
        address=_AUDIT_ADDR,
        budget_usd=50.0,
        tools=[
            "generate_answer",
            "search_context",
            "view_metrics",
            "access_audit_log",
            "generate_report",
        ],
        clearance="secret",
        description="Compliance audit — full audit log + reports",
    ),
]
_TIER_BY_ROLE: dict[str, CapstoneTier] = {t.role: t for t in CAPSTONE_TIERS}


def _attach_envelopes(engine: "GovernanceEngine") -> None:
    """Attach a full ConstraintEnvelopeConfig to every agent role address.

    Without an envelope, `engine.verify_action()` on a role auto-approves
    ("No envelope constraints -- action permitted"). The envelope is the
    source of restriction.
    """
    from pact import (
        CommunicationConstraintConfig,
        ConfidentialityLevel,
        ConstraintEnvelopeConfig,
        DataAccessConstraintConfig,
        FinancialConstraintConfig,
        OperationalConstraintConfig,
        RoleEnvelope,
        TemporalConstraintConfig,
    )

    for tier in CAPSTONE_TIERS:
        envelope = ConstraintEnvelopeConfig(
            id=f"{tier.role}_envelope",
            description=tier.description,
            confidentiality_clearance=ConfidentialityLevel(tier.clearance),
            financial=FinancialConstraintConfig(max_spend_usd=tier.budget_usd),
            operational=OperationalConstraintConfig(
                allowed_actions=list(tier.tools),
                blocked_actions=[],
            ),
            temporal=TemporalConstraintConfig(blackout_periods=[]),
            data_access=DataAccessConstraintConfig(
                read_paths=["/public/*"],
                write_paths=[],
                blocked_data_types=[],
            ),
            communication=CommunicationConstraintConfig(allowed_channels=["internal"]),
            max_delegation_depth=3,
        )
        engine.set_role_envelope(
            RoleEnvelope(
                id=f"{tier.role}_role_envelope",
                defining_role_address=_DIRECTOR_ADDR,
                target_role_address=tier.address,
                envelope=envelope,
            )
        )


def build_capstone_stack(
    engine: "GovernanceEngine",
) -> tuple[dict[str, GovernedSupervisor], list[CapstoneTier]]:
    """Build the shared 3-tier GovernedSupervisor stack.

    Returns:
        ``(agents_by_role, tiers)`` — ``agents_by_role`` maps
        ``"qa" | "admin" | "audit"`` to its ``GovernedSupervisor`` (the shape
        ``handle_qa`` expects); ``tiers`` is the ``CapstoneTier`` metadata.

    Side effect: attaches a ``ConstraintEnvelopeConfig`` to every agent role
    address on ``engine``.
    """
    _attach_envelopes(engine)

    agents_by_role: dict[str, GovernedSupervisor] = {}
    for tier in CAPSTONE_TIERS:
        agents_by_role[tier.role] = GovernedSupervisor(
            model=MODEL,
            budget_usd=tier.budget_usd,
            tools=list(tier.tools),
            data_clearance=tier.clearance,
        )
    return agents_by_role, list(CAPSTONE_TIERS)


# ════════════════════════════════════════════════════════════════════════
# SHARED QA HANDLER — used by Nexus deployment AND monitoring/test files
# ════════════════════════════════════════════════════════════════════════
#
# `handle_qa()`:
#   1. Refuses an unknown role (never falls back to another tier).
#   2. If an engine is passed, asks `engine.verify_action(tier address,
#      action)` first — a blocked verdict is returned as blocked.
#   3. Runs the tier's `GovernedSupervisor.run(objective, execute_node=...)`
#      whose executor calls the shared `CapstoneQAAgent` on local Ollama.
#   4. A node HELD by the budget envelope is returned as blocked; a FAILED
#      node (LLM error) RAISES — there is no offline stub.

_shared_qa_agent: CapstoneQAAgent | None = None


def _get_shared_agent() -> CapstoneQAAgent:
    global _shared_qa_agent
    if _shared_qa_agent is None:
        _shared_qa_agent = CapstoneQAAgent(CapstoneQAConfig())
    return _shared_qa_agent


async def _capstone_execute_node(_spec: Any, inputs: dict[str, Any]) -> dict[str, Any]:
    """Executor callback for GovernedSupervisor.run() — a real LLM call.

    Returns ``{"result": <agent output dict>, "cost": 0.0}``. Cost is zero
    because local Ollama is free. Raises if the agent reports an error, so
    the supervisor marks the node FAILED and ``handle_qa`` surfaces it.
    """
    # GovernedSupervisor puts the objective in the plan node's AgentSpec
    # (``spec.description``); ``inputs`` only carries resolved upstream
    # outputs and is EMPTY for the root task. Reading only ``inputs`` sent the
    # model the literal string "{}".
    objective = (
        getattr(_spec, "description", None)
        or inputs.get("objective")
        or inputs.get("question")
        or inputs.get("prompt")
    )
    if not objective:
        raise ValueError("executor received no objective (spec.description and inputs empty)")
    out = await _get_shared_agent().run_async(question=str(objective))
    if not isinstance(out, dict) or out.get("error"):
        detail = out.get("error") if isinstance(out, dict) else repr(out)
        raise RuntimeError(f"CapstoneQAAgent failed: {detail}")
    if not str(out.get("answer", "")).strip():
        raise RuntimeError(f"CapstoneQAAgent returned no answer: {out!r}")
    return {"result": out, "cost": 0.0}


def _as_float(value: Any) -> float | None:
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


async def handle_qa(
    question: str,
    role: str,
    agents_by_role: dict[str, GovernedSupervisor],
    *,
    engine: "GovernanceEngine | None" = None,
    action: str = "generate_answer",
) -> dict[str, Any]:
    """Route a question to the governed supervisor for ``role``.

    Returns on success::

        {"answer": str, "confidence": float | None, "sources": list[str],
         "reasoning_steps": list[str], "latency_ms": float,
         "budget_consumed": float, "governed": True, "blocked": False,
         "verdict": "served", "role": str}

    ``confidence`` is the agent's own self-reported value (None if it did
    not return a number). On a governance refusal (unknown role, blocked
    verdict, budget HELD) returns ``{"error", "blocked": True, "verdict",
    "governed": True, "role"}``. An LLM failure raises ``RuntimeError``.
    """
    if role not in agents_by_role or role not in _TIER_BY_ROLE:
        return {
            "error": f"unknown role {role!r} — refused",
            "blocked": True,
            "verdict": "unknown_role",
            "governed": True,
            "role": role,
        }
    tier = _TIER_BY_ROLE[role]
    if engine is not None:
        verdict = engine.verify_action(tier.address, action, {"cost": 0.0})
        if not verdict.allowed:
            return {
                "error": verdict.reason,
                "blocked": True,
                "verdict": verdict.level,
                "governed": True,
                "role": role,
            }

    gs = agents_by_role[role]
    start = time.perf_counter()
    result = await gs.run(objective=question, execute_node=_capstone_execute_node)
    latency_ms = (time.perf_counter() - start) * 1000

    if not result.success:
        nodes = list(result.plan.nodes.values()) if result.plan else []
        if any(n.state.name == "HELD" for n in nodes):
            return {
                "error": "budget envelope exhausted — request held",
                "blocked": True,
                "verdict": "held",
                "governed": True,
                "role": role,
            }
        errors = [n.error for n in nodes if n.error]
        raise RuntimeError(
            f"capstone '{role}' tier: LLM call failed: {errors}. "
            "Start Ollama: ollama serve"
        )

    out = next(iter(result.results.values()))
    return {
        "answer": str(out.get("answer", "")),
        "confidence": _as_float(out.get("confidence")),
        "sources": list(out.get("sources") or []),
        "reasoning_steps": list(out.get("reasoning_steps") or []),
        "latency_ms": latency_ms,
        "budget_consumed": result.budget_consumed,
        "governed": True,
        "blocked": False,
        "verdict": "served",
        "role": role,
    }


# ════════════════════════════════════════════════════════════════════════
# RUN HELPER — for technique files that use asyncio at module scope
# ════════════════════════════════════════════════════════════════════════


def run_async(coro):  # noqa: ANN001 — coroutine
    """Run a coroutine to completion from synchronous code.

    In a plain script there is no running loop, so ``asyncio.run`` is used.
    Inside Jupyter/Colab a loop is already running, so the coroutine runs on
    a fresh loop in a worker thread. Exceptions from the coroutine propagate
    unchanged.
    """
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coro).result()




# ════════════════════════════════════════════════════════════════════════
# MLFP06 — Exercise 8.1: Load a Fine-Tuned Adapter via AdapterRegistry
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Use kailash-align's AdapterRegistry as the source of truth for
#     model provenance (base model, method, LoRA config, version)
#   - Register the adapters Ex 2.6 (SFT) and Ex 3.3 (DPO) trained, and
#     select one by a named preference — never by a hardcoded path
#   - Load the selected adapter for inference (AdapterMerger + the HF
#     generation backend) and score it on a few MMLU questions
#   - Visualise real adapter sizes against the base model
#   - Apply adapter loading to a Singapore HR compliance scenario
#
# PREREQUISITES: MLFP06 Ex 2.6 (SFT adapter), Ex 3.3 (DPO adapter) — this
#   file loads what those runs saved; it stops with instructions if none
# ESTIMATED TIME: ~25 min
#
# TASKS:
#   1. Load MMLU evaluation data
#   2. Register the trained adapters and pick the best available one
#   3. Merge the adapter into its base model and generate answers
#   4. Visualise the adapter catalogue and parameter counts
#   5. Apply to a Singapore HR compliance QA scenario
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import re

import matplotlib.pyplot as plt
import polars as pl
from kailash_align import AdapterMerger, AdapterRegistry, AdapterSignature
from kailash_align.exceptions import AdapterNotFoundError
from kailash_align.vllm_backend import HFGenerationBackend



## THEORY — Why an AdapterRegistry?


In [ ]:
# A production LLM platform trains many adapters: SFT for domain tone,
# DPO for preference alignment, merges that combine both. The
# AdapterRegistry is version control for those weights — it records
# which base model each adapter attaches to, how it was trained (LoRA
# rank, alpha, target modules), its metrics, and its version.
#
# Hardcoding a path like "./models/imdb_sft_v1" looks fine today and
# silently ships the wrong weights tomorrow. The registry turns model
# loading into a named lookup against a catalogue.
#
# One honest caveat: `AdapterRegistry()` built without a backing model
# registry lives in memory, so a new process starts empty. The adapters
# Ex 2.6 / 3.3 trained are still on disk (AlignmentPipeline saves them to
# <experiment_dir>/<adapter_name>/<method>/adapter/), so this file finds
# them there and registers them before looking anything up.



## TASK 1 — Load MMLU evaluation data


In [ ]:
eval_data = load_mmlu_eval(n_rows=100)

print(f"\nEvaluation data (MMLU): {eval_data.shape}")
print(f"Subjects: {eval_data['subject'].n_unique()}")



### Checkpoint 1


In [ ]:
assert eval_data.height > 0, "Task 1: MMLU should load at least 1 row"
assert "instruction" in eval_data.columns, "Task 1: expected 'instruction' column"
print("✓ Checkpoint 1 passed — evaluation data loaded\n")



## TASK 2 — Register the trained adapters, pick the best available one


Register every on-disk adapter, then select by named preference.


In [ ]:
on_disk = discover_trained_adapters()
if not on_disk:
    raise FileNotFoundError(
        "No trained adapters found under "
        f"{[str(r) for r in ADAPTER_SEARCH_ROOTS]}. Run "
        "modules/mlfp06/solutions/ex_2/06_sft_alignment_pipeline.py and "
        "ex_3/03_dpo_training.py first (from the repo root)."
    )

# Preferred order: the DPO-aligned adapter, then the SFT adapter.
PREFERENCE = ("ultrafeedback_dpo_v1", "imdb_sentiment_sft_v1")


async def register_and_pick() -> tuple[AdapterRegistry, object]:
    registry = AdapterRegistry()
    for found in on_disk:
        await registry.register_adapter(
            name=found["adapter_name"],
            adapter_path=found["adapter_path"],
            signature=AdapterSignature(
                base_model_id=found["base_model_id"],
                rank=found["rank"],
                alpha=found["alpha"],
                target_modules=found["target_modules"],
                training_method=found["method"],
            ),
            tags=[found["method"]],
        )

    versions = await registry.list_adapters()
    print(f"Registered adapters: {len(versions)}")
    for av in versions:
        print(
            f"  {av.adapter_name:28s} v{av.version}  base={av.base_model_id}  "
            f"stage={av.stage}"
        )

    for name in PREFERENCE + tuple(av.adapter_name for av in versions):
        try:
            return registry, await registry.get_adapter(name)
        except AdapterNotFoundError:
            print(f"  (no adapter named {name!r} — trying the next preference)")
    raise AdapterNotFoundError("registry is empty after registration")


registry, best = run_async(register_and_pick())



### Checkpoint 2


In [ ]:
assert best.adapter_name, "Task 2: an adapter must be selected"
assert best.base_model_id, "Task 2: the adapter must name its base model"
print(
    f"✓ Checkpoint 2 passed — selected {best.adapter_name} v{best.version} "
    f"(base {best.base_model_id})\n"
)



## TASK 3 — Load the adapter for inference and generate answers


In [ ]:
# AdapterMerger loads the base model, applies the LoRA adapter and
# merges the weights (merge_and_unload), then records merge_status and
# merged_model_path back in the registry. HFGenerationBackend then
# generates from the merged model on CPU/MPS/CUDA. (For an Ollama
# deployment, AlignmentServing.deploy_ollama goes one step further and
# exports a GGUF; that needs llama.cpp tooling and is not run here.)

merged_path = run_async(AdapterMerger(adapter_registry=registry).merge(best.adapter_name))
backend = HFGenerationBackend(model_id=str(merged_path))

N_EVAL = 5
sample = eval_data.head(N_EVAL)
prompts = [
    f"{q}\n\nAnswer with a single letter (A, B, C or D).\nAnswer:"
    for q in sample["instruction"].to_list()
]
completions = backend.batch_generate(prompts, max_new_tokens=4, temperature=0.0)
predicted = []
for c in completions:
    m = re.search(r"\b([ABCD])\b", c[0])
    predicted.append(m.group(1) if m else "?")
mmlu_acc = sum(p == g for p, g in zip(predicted, sample["response"].to_list())) / N_EVAL
backend.shutdown()

print(f"Merged model: {merged_path}")
for subj, p, g in zip(sample["subject"].to_list(), predicted, sample["response"].to_list()):
    print(f"  {subj:<32s} predicted={p}  gold={g}")
print(f"MMLU accuracy on {N_EVAL} questions: {mmlu_acc:.0%} (chance = 25%)")



### Checkpoint 3


In [ ]:
assert len(completions) == N_EVAL, "Task 3: one completion per prompt"
assert merged_path.exists(), "Task 3: the merged model should be on disk"
print("✓ Checkpoint 3 passed — adapter loaded and generating\n")
# INTERPRETATION: 5 questions is a smoke test, not a benchmark — one
# right or wrong answer moves accuracy by 20 points. The point is that
# the weights you serve are the registered ones, provably.



## TASK 4 — Visualise the adapter catalogue


In [ ]:
catalogue = pl.DataFrame(
    {
        "adapter_name": [f["adapter_name"] for f in on_disk],
        "method": [f["method"] for f in on_disk],
        "base_model_id": [f["base_model_id"] for f in on_disk],
        "lora_rank": [f["rank"] for f in on_disk],
        "target_modules": [",".join(f["target_modules"]) for f in on_disk],
        "trainable_params": [f["trainable_params"] for f in on_disk],
    }
)
catalogue.write_parquet(OUTPUT_DIR / "adapter_catalogue.parquet")
print("Adapter catalogue:")
print(catalogue)

# INTERPRETATION: The catalogue is the single pane of glass operations
# teams use for rollback. If a new adapter ships broken, they pick the
# previous row and redeploy by name.



## VISUALISE — Adapter parameter count vs the base model


In [ ]:
# Both numbers are counted from the safetensors files on disk: the
# adapter's own weights, and the merged model (= base model size).

base_params = count_safetensors_params(merged_path)
names = catalogue["adapter_name"].to_list()
param_counts = catalogue["trainable_params"].to_list()

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(names, param_counts, color=["#3498db", "#2ecc71", "#e67e22"][: len(names)])
ax.axhline(
    base_params,
    color="#e74c3c",
    linestyle="--",
    linewidth=2,
    label=f"Base model ({best.base_model_id}): {base_params / 1e6:,.0f}M params",
)
ax.set_ylabel("Parameters")
ax.set_title("Adapter Parameters vs Base Model (counted from disk)", fontweight="bold")
ax.set_yscale("log")
for bar, count in zip(bars, param_counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        max(count, 1) * 1.5,
        f"{count / 1e6:.2f}M\n({count / base_params:.2%})",
        ha="center",
        fontsize=9,
    )
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3, which="both")
plt.tight_layout()
fname = OUTPUT_DIR / "ex8_adapter_params.png"
plt.savefig(fname, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"\n  Saved: {fname}")



## TASK 5 — Apply: Singapore HR Compliance QA


Base model:     {best.base_model_id}
  Active adapter: {best.adapter_name} v{best.version}
  LoRA config:    r={best.lora_config.get('r')}, alpha={best.lora_config.get('alpha')}, targets={best.lora_config.get('target_modules')}
  Smoke-test MMLU accuracy: {mmlu_acc:.0%} on {N_EVAL} questions

  Illustrative legal-review baseline: S${ILLUSTRATIVE_REVIEW_COST_SGD * ILLUSTRATIVE_QUERIES_PER_MONTH:,}/month
  ({ILLUSTRATIVE_QUERIES_PER_MONTH} questions at S${ILLUSTRATIVE_REVIEW_COST_SGD} each)

  Audit trail: log adapter name + version with every answer, so an
  inspection can reconstruct which model answered any policy question.


In [ ]:
# SCENARIO: A Singapore SME (200 employees) runs an internal HR policy
# assistant tuned on Ministry of Manpower guidance (CPF, Work Pass
# categories, retrenchment notice). DPO alignment then discourages
# speculative or legally risky phrasing.
#
# BUSINESS IMPACT (illustrative figures): if external legal review of an
# HR question costs ~S$400, 200 questions a month cost S$80,000. A
# governed adapter answering first-line questions at a few cents each,
# with lawyers reviewing only escalations, cuts most of that — and the
# registry lets the HR team show which adapter version answered which
# question.

ILLUSTRATIVE_REVIEW_COST_SGD = 400
ILLUSTRATIVE_QUERIES_PER_MONTH = 200
print("\n" + "=" * 70)
print("  APPLY — Singapore HR Policy Assistant")
print("=" * 70)
print(
)



## REFLECTION


[x] Registered trained adapters in AdapterRegistry with their signatures
  [x] Selected an adapter by named preference, not by a hardcoded path
  [x] Merged the adapter into its base model and generated real answers
  [x] Counted adapter vs base parameters from the files on disk
  [x] Applied adapter provenance to a Singapore HR compliance scenario

  KEY INSIGHT: A model you cannot name, version and trace to its
  training run is a model you cannot roll back or defend in an audit.

  Next: 02_governance_pipeline.py wraps the served model in PACT
  governance tiers.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

